# Grid Runner — Kaggle (P100)

Runs **AdaCAL** priority experiments on the Kaggle P100 GPU.

**Setup instructions:**
1. Upload the repo as a Kaggle Dataset (or use the public dataset link).
2. Add your W&B API key as a Kaggle Secret named `WANDB_API_KEY`.
3. Enable Internet access in Settings if needed for W&B logging.
4. Run all cells top-to-bottom.

Results are saved to `/kaggle/working/results/` and can be downloaded via the Output tab.

In [ ]:
# ── Cell 1: Setup ──────────────────────────────────────────────────────────
# Install dependencies (no git clone needed — repo is a Kaggle Dataset)
!pip install torch aeon wfdb wandb tqdm pyyaml scikit-learn --quiet

import sys, os

# Adjust this path to match your Kaggle dataset mount point
# e.g. /kaggle/input/adaptive-imbalance-tsc/adaptive-imbalance-tsc
REPO_DIR = "/kaggle/input/adaptive-imbalance-tsc"

# Add repo to Python path
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# Working dir: writable area
WORK_DIR = "/kaggle/working"
os.chdir(WORK_DIR)

# Copy scripts/configs to working dir so we can run them
import shutil
for subdir in ["scripts", "configs", "src"]:
    src_path = os.path.join(REPO_DIR, subdir)
    dst_path = os.path.join(WORK_DIR, subdir)
    if os.path.isdir(src_path) and not os.path.exists(dst_path):
        shutil.copytree(src_path, dst_path)

# Ensure working dir is on path
if WORK_DIR not in sys.path:
    sys.path.insert(0, WORK_DIR)

print("Setup complete. Working dir:", os.getcwd())
print("Python path:", sys.path[:3])

In [ ]:
# ── Cell 2: W&B API key from Kaggle Secrets ────────────────────────────────
# Add your key at: https://www.kaggle.com/settings → API → Secrets
# Secret name: WANDB_API_KEY

import os

try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    WANDB_API_KEY = user_secrets.get_secret("WANDB_API_KEY")
    os.environ["WANDB_API_KEY"] = WANDB_API_KEY
    !wandb login {WANDB_API_KEY} --relogin --quiet
    print("W&B API key loaded from Kaggle Secrets.")
except Exception as e:
    print(f"W&B not configured ({e}). Experiments will run without W&B logging.")

In [ ]:
# ── Cell 3: Filter selector ────────────────────────────────────────────────
# AdaCAL gets priority on the P100 — change if you want to run other methods.

METHOD_FILTER  = ['adacal']   # AdaCAL priority on P100
DATASET_FILTER = []           # empty = all datasets
ARCH_FILTER    = []           # empty = all architectures

print("Methods  :", METHOD_FILTER  or "ALL")
print("Datasets :", DATASET_FILTER or "ALL")
print("Archs    :", ARCH_FILTER    or "ALL")

In [ ]:
# ── Cell 4: Generate configs + run grid ───────────────────────────────────
import os
os.chdir("/kaggle/working")

RESULTS_DIR = "/kaggle/working/results"
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs("configs/experiments", exist_ok=True)

# Generate all 324 configs into working dir
!python scripts/generate_configs.py --output_dir configs/experiments

# Build filter CLI args
filter_args = ""
if METHOD_FILTER:
    filter_args += " --filter_method " + " ".join(METHOD_FILTER)
if DATASET_FILTER:
    filter_args += " --filter_dataset " + " ".join(DATASET_FILTER)
if ARCH_FILTER:
    filter_args += " --filter_arch " + " ".join(ARCH_FILTER)

cmd = (
    f"python scripts/run_grid.py "
    f"--config_dir configs/experiments "
    f"--results_dir {RESULTS_DIR} "
    f"--skip_completed "
    f"{filter_args}"
)
print("Running:", cmd)
!{cmd}

In [ ]:
# ── Cell 5: Save results to /kaggle/working/results/ ──────────────────────
# Results are already in RESULTS_DIR. Consolidate into a single CSV.

import os
os.chdir("/kaggle/working")

!python scripts/consolidate_results.py \
    --results_dir /kaggle/working/results \
    --output /kaggle/working/raw_results.csv

print("\nAll output files are in /kaggle/working/ and can be downloaded via the Output tab.")
!ls -lh /kaggle/working/raw_results.csv 2>/dev/null || echo "raw_results.csv not yet generated."

In [ ]:
# ── Cell 6: Instructions for downloading results ───────────────────────────
print("""
=== Downloading Results ===

Option A — Kaggle Output tab:
  1. Go to your notebook's Output tab.
  2. Download /kaggle/working/raw_results.csv and /kaggle/working/results/

Option B — Kaggle API:
  kaggle kernels output <your-username>/<notebook-slug> -p ./local_results/

Option C — W&B dashboard (if W&B was configured):
  Visit https://wandb.ai and find project adaptive-imbalance-tsc.
""")